In [1]:
import torch
print(torch.__version__)

2.11.0+cu128


In [2]:
!pip install -q -U transformers accelerate bitsandbytes
import torch
print(torch.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00
2.11.0+cu128


In [3]:
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

2.11.0+cu128
CUDA available: True


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [5]:
class QuantizedLLMEngine:
    def __init__(self, model_id: str):
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=quant_config,
            device_map="auto",
        )
        self.model.eval()

    @torch.no_grad()
    def generate_from_messages(self, messages, gen_kwargs):
        inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to(self.model.device)

        input_len = inputs["input_ids"].shape[-1]

        outputs = self.model.generate(**inputs, **gen_kwargs)

        new_tokens = outputs[:, input_len:]

        response = self.tokenizer.decode(
            new_tokens[0],
            skip_special_tokens=True
        ).strip()

        del inputs
        del outputs
        del new_tokens
        torch.cuda.empty_cache()

        return response

In [6]:
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
engine = QuantizedLLMEngine(MODEL_ID)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [7]:
print(engine.model.get_memory_footprint() / 1e9, "GB")

2.206341504 GB


In [8]:
test_messages = [
    {"role": "user", "content": "In one sentence, what is the capital of France?"}
]
test_gen_kwargs = {"max_new_tokens": 30, "do_sample": False}
response = engine.generate_from_messages(test_messages, test_gen_kwargs)
print(response)

The capital of France is Paris.


In [9]:
class BaseAgent:
    # ... unchanged, leave exactly as-is ...
    def __init__(self, engine, system_prompt, gen_kwargs):
        self.engine = engine
        self.system_prompt = system_prompt
        self.gen_kwargs = gen_kwargs
        self.history = [{"role": "system", "content": self.system_prompt}]

    def respond(self, user_turn):
        self.history.append({"role": "user", "content": user_turn})
        reply = self.engine.generate_from_messages(self.history, self.gen_kwargs)
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def reset(self):
        self.history = [{"role": "system", "content": self.system_prompt}]


class ProposerAgent(BaseAgent):
    def __init__(self, engine):
        system_prompt = (
            "You are a reasoning agent. Given a problem, propose a solution "
            "with your full step-by-step reasoning shown. Do not use LaTeX or "
            "special math notation — write all math in plain text (e.g. '2x + 1 = 3', "
            "not '\\[2x+1=3\\]'). If you receive a critique of a previous answer, "
            "revise your reasoning to directly address the specific objection raised — "
            "do not simply restate your original answer.\n\n"
            "You MUST end your entire response with a line in exactly this format, "
            "with nothing after it:\n"
            "FINAL ANSWER: <your answer here>"
        )
        gen_kwargs = {"max_new_tokens": 400, "do_sample": True, "temperature": 0.8, "top_p": 0.95}
        super().__init__(engine, system_prompt, gen_kwargs)


class CriticAgent(BaseAgent):
    def __init__(self, engine):
        system_prompt = (                                              # <-- THIS is the part you replace
            "Your role is ONLY to verify the Proposer's reasoning. Do not solve the "
            "problem independently, derive the correct answer, provide an alternative "
            "solution, or state a final numeric or textual answer of your own. "
            "If the Proposer is wrong, identify only the specific flaw in its reasoning.\n\n"
            "Respond in EXACTLY this format and nothing else:\n"
            "Decision: APPROVED\n"
            "or\n"
            "Decision: REJECTED\n"
            "Flaw: <the specific error in the Proposer's reasoning, one or two sentences>\n\n"
            "Do not include any text after the Flaw line. Do not restate or solve the problem."
        )
        gen_kwargs = {"max_new_tokens": 256, "do_sample": False}
        super().__init__(engine, system_prompt, gen_kwargs)

In [10]:
proposer = ProposerAgent(engine)

problem = "A farmer has 17 sheep, and all but 9 die. How many sheep does the farmer have left?"
answer = proposer.respond(problem)
print(answer)

To solve this problem, we need to understand what the phrase "all but 9 die" means. In this context, "all but" refers to all the sheep except for 9.


1. The farmer initially has 17 sheep.

2. The phrase "all but 9 die" indicates that 9 sheep are alive.

3. Therefore, the number of sheep the farmer has left is 9.


FINAL ANSWER: The farmer has 9 sheep left.


In [11]:
critic = CriticAgent(engine)

critique = critic.respond(
    f"Problem: {problem}\nProposed solution: {answer}"
)
print(critique)

Decision: APPROVED
Flaw: There is no flaw in the Proposer's reasoning.


In [12]:
problem2 = (
    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
    "How much does the ball cost?"
)
answer2 = proposer.respond(problem2)
print(answer2)

To solve this problem, let's denote the cost of the ball as B and the cost of the bat as B + $1.00 (since the bat costs $1.00 more than the ball).


1. We know the total cost of the bat and the ball is $1.10.

2. So we can write the equation: B + (B + $1.00) = $1.10.

3. Simplifying the equation: 2B + $1.00 = $1.10.

4. Subt0ract $1.00 from both sides to isolate the variable term: 2B = $0.10.

5. Divide both sides by 2 to solve for B: B = $0.05.


Therefore, the ball costs $0.05.


FINAL ANSWER: The ball costs $0.05.


In [13]:
fake_wrong_answer = "The ball costs $0.10, since $1.10 - $1.00 = $0.10."

critique2 = critic.respond(
    f"Problem: {problem2}\nProposed solution: {fake_wrong_answer}"
)
print(critique2)

Decision: REJECTED
Flaw: The Proposer's reasoning is incorrect because it does not consider the possibility that the ball could cost $0.05 and the bat $1.05. The correct approach is to set up an equation where the cost of the ball is x and the cost of the bat is x + $1.00. The equation would be x + (x + $1.00) = $1.10. Solving this equation gives us x = $0.05.


In [14]:
import re

def extract_conclusion(proposer_text: str) -> str:
    text = proposer_text.lower()
    if "final answer:" in text:
        conclusion = text.split("final answer:", 1)[1]
    elif "final answer" in text:
        conclusion = text.split("final answer", 1)[1]
    elif "therefore" in text:
        conclusion = text.split("therefore", 1)[1]
    else:
        sentence_parts = re.split(r'\.(?!\d)', text)
        sentences = [s for s in sentence_parts if s.strip()]
        conclusion = sentences[-1] if sentences else text

    return conclusion.strip(" ,.:\n")

In [15]:
test_broken_case = "so, the ball costs $0.05."
print(extract_conclusion(test_broken_case))

print(extract_conclusion(answer))
print(extract_conclusion(answer2))
print(extract_conclusion("The problem involves careful accounting. We add the two quantities. The total comes out to 42."))

so, the ball costs $0.05
the farmer has 9 sheep left
the ball costs $0.05
the total comes out to 42


In [16]:
def check_for_cycle(current_conclusion: str, was_approved: bool, conclusion_history: list) -> bool:
    """
    conclusion_history: list of (conclusion, was_approved) tuples from prior rounds.
    Return True only if current_conclusion matches a PRIOR conclusion that was
    also NOT approved — i.e. genuine unresolved oscillation, not convergence-via-repeat.
    """

    for previous_conclusion, previous_approved in conclusion_history:
        if previous_conclusion == current_conclusion and not previous_approved:
            return True

    return False

In [17]:
history_test = [("the ball costs 5 cents", False), ("the ball costs 10 cents", False)]

print(check_for_cycle("the ball costs 5 cents", False, history_test))   # expect True — repeat of a rejected one
print(check_for_cycle("the ball costs 20 cents", False, history_test))  # expect False — brand new conclusion
history_test2 = [("the ball costs 5 cents", True)]
print(check_for_cycle("the ball costs 5 cents", False, history_test2))  # expect False — repeat of an APPROVED one, healthy convergence

True
False
False


In [18]:
class ConsensusOrchestrator:
    def __init__(self, proposer: ProposerAgent, critic: CriticAgent, nli_checker,
                 soft_min_rounds: int = 2, hard_max_rounds: int = 6):
        self.proposer = proposer
        self.critic = critic
        self.nli_checker = nli_checker
        self.soft_min_rounds = soft_min_rounds
        self.hard_max_rounds = hard_max_rounds
        self.conclusion_history = []

    def run(self, problem: str) -> dict:
        self.proposer.reset()
        self.critic.reset()
        self.conclusion_history = []
        proposer_text = self.proposer.respond(problem)
        trace = []

        for round_num in range(self.hard_max_rounds):
            conclusion = extract_conclusion(proposer_text)
            critic_text = self.critic.respond(
                f"Problem: {problem}\nProposed solution: {proposer_text}"
            )
            critic_approved = "Decision: APPROVED" in critic_text

            nli_contradiction = None
            if self.conclusion_history:
                prev_conclusion, _ = self.conclusion_history[-1]
                nli_contradiction = self.nli_checker(prev_conclusion, conclusion)

            if critic_approved and not nli_contradiction:
                final_status = "approved"
            elif (not critic_approved) and nli_contradiction:
                final_status = "rejected"
            else:
                final_status = "unresolved"

            cycle_detected = self.check_for_cycle_and_break(conclusion, critic_approved)

            trace.append({
                "round": round_num, "proposer_text": proposer_text, "conclusion": conclusion,
                "critic_approved": critic_approved, "nli_contradiction": nli_contradiction,
                "final_status": final_status, "cycle_detected": cycle_detected,
            })

            if final_status == "approved":
                return {"final_answer": conclusion, "rounds_used": round_num + 1,
                        "trace": trace, "stopped_reason": "approved"}

            self.conclusion_history.append((conclusion, critic_approved))

            past_soft_minimum = round_num + 1 >= self.soft_min_rounds
            if past_soft_minimum and cycle_detected:
                return {"final_answer": conclusion, "rounds_used": round_num + 1,
                        "trace": trace, "stopped_reason": "cycle_detected"}

            if past_soft_minimum and nli_contradiction is False:
                return {"final_answer": conclusion, "rounds_used": round_num + 1,
                        "trace": trace, "stopped_reason": "converged_without_approval"}

            proposer_text = self.proposer.respond(
                f"Critic feedback: {critic_text}\nPlease revise your reasoning."
            )

        last_conclusion, _ = self.conclusion_history[-1]
        return {"final_answer": last_conclusion, "rounds_used": self.hard_max_rounds,
                "trace": trace, "stopped_reason": "hard_max_rounds_exhausted"}

    def check_for_cycle_and_break(self, current_conclusion: str, was_approved: bool) -> bool:
        return check_for_cycle(current_conclusion, was_approved, self.conclusion_history)

In [19]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer as NLITokenizer

NLI_MODEL_ID = "roberta-large-mnli"

nli_tokenizer = NLITokenizer.from_pretrained(NLI_MODEL_ID)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_ID)
nli_model.eval()

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
 

In [20]:
print(nli_model.config.id2label)

{0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}


In [21]:
@torch.no_grad()
def nli_checker(premise: str, hypothesis: str) -> bool:
    inputs = nli_tokenizer(
        premise, hypothesis,
        return_tensors="pt", truncation=True
    ).to(nli_model.device)
    logits = nli_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    contradiction_prob = probs[0].item()
    return contradiction_prob > 0.5

In [22]:
print(nli_checker("the ball costs $0.05", "the ball costs $0"))  # expect True — real contradiction
print(nli_checker("the ball costs $0.05", "the ball costs 5 cents"))  # expect False — same meaning, different words

True
False


In [23]:
orchestrator = ConsensusOrchestrator(proposer, critic, nli_checker, soft_min_rounds=2, hard_max_rounds=5)
result7 = orchestrator.run(problem2)
print(result7["rounds_used"], result7["stopped_reason"])
for entry in result7["trace"]:
    print(entry["conclusion"], "|", entry["final_status"], "|", entry["nli_contradiction"])

2 approved
the ball costs $0.05 | unresolved | None
calculate the value of \( x \):
\[ x = 0.20 \]

this result shows that if the bat costs less than $1.00 | approved | False


In [24]:
result4 = orchestrator.run(problem2)

for entry in result4["trace"]:
    print("Round:", entry["round"])
    print("Proposer text:", entry["proposer_text"])
    print("Extracted conclusion:", entry["conclusion"])
    print("---")

Round: 0
Proposer text: Let's denote the cost of the ball as 'x'. According to the problem, the bat costs $1.00 more than the ball, which we can express as 'x + $1.00'.


The total cost of the bat and the ball is $1.10. We can write this as an equation:


x (cost of the ball) + (x + $1.00) (cost of the bat) = $1.10


Combining like terms gives us:


2x + $1.00 = $1.10


Subt0racting $1.00 from both sides of the equation, we get:


2x = $0.10


Now we divide both sides by 2 to solve for 'x':


x = $0.10 / 2

x = $0.05


FINAL ANSWER: The ball costs $0.05.
Extracted conclusion: the ball costs $0.05
---


In [25]:
print(result4["trace"][0].keys())

dict_keys(['round', 'proposer_text', 'conclusion', 'critic_approved', 'nli_contradiction', 'final_status', 'cycle_detected'])


In [26]:
orchestrator = ConsensusOrchestrator(proposer, critic, nli_checker, soft_min_rounds=2, hard_max_rounds=5)

In [27]:
result5 = orchestrator.run(problem2)
for entry in result5["trace"]:
    print(entry)

{'round': 0, 'proposer_text': "Let's denote the cost of the ball as x dollars. According to the problem, the bat costs $1.00 more than the ball, so the cost of the bat is x + $1.00.\n\nThe total cost of the bat and the ball is $1.10. So, we can write the equation:\n\nx (cost of the ball) + (x + $1.00) (cost of the bat) = $1.10\n\nCombining like terms, we get:\n\n2x + $1.00 = $1.10\n\nSubtract $1.00 from both sides to isolate the term with x:\n\n2x = $1.10 - $1.00\n2x = $0.10\n\nNow, divide both sides by 2 to solve for x:\n\nx = $0.10 / 2\nx = $0.05\n\nTherefore, the ball costs $0.05.\n\nFINAL ANSWER: $0.05", 'conclusion': '$0.05', 'critic_approved': True, 'nli_contradiction': None, 'final_status': 'approved', 'cycle_detected': False}


In [28]:
!pip install -q -U sentence-transformers
from sentence_transformers import SentenceTransformer, util
embedder = SentenceTransformer("all-MiniLM-L6-v2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 44.2 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [29]:
def semantic_diversity(conclusion_a: str, conclusion_b: str) -> float:
    """
    Returns a diversity score between 0 and ~1+: higher = more semantically different.
    """
    embeddings = embedder.encode([conclusion_a, conclusion_b])
    similarity = util.cos_sim(embeddings[0], embeddings[1]).item()
    diversity = 1 - similarity
    return diversity

In [30]:
print(semantic_diversity("the ball costs $0.05", "the ball costs $0"))
print(semantic_diversity("the ball costs $0.05", "the ball costs 5 cents"))

0.0807538628578186
0.1252160668373108


In [31]:
test_response = proposer.respond(problem2)
print(len(test_response))
print(test_response)

561
Let's denote the cost of the ball as x dollars. According to the problem, the bat costs $1.00 more than the ball, so the cost of the bat is x + $1.00.

The total cost of the bat and the ball is $1.10. So, we can write the equation:

x (cost of the ball) + (x + $1.00) (cost of the bat) = $1.10

Combining like terms, we get:

2x + $1.00 = $1.10

Subt0ract $1.00 from both sides to isolate the term with x:

2x = $1.10 - $1.00
2x = $0.10

Now, divide both sides by 2 to solve for x:

x = $0.10 / 2
x = $0.05

Therefore, the ball costs $0.05.

FINAL ANSWER: $0.05


In [32]:
result8 = orchestrator.run(problem2)
print(result8["rounds_used"], result8["stopped_reason"])
for entry in result8["trace"]:
    print(entry["conclusion"], "|", entry["final_status"], "|", entry["nli_contradiction"])

2 converged_without_approval
the ball costs $0.05 | unresolved | None
simplifying the equation:
   x + x - $1.00 = $ | unresolved | False


In [33]:
print(engine)
print(proposer)
print(critic)
print(nli_checker)
print(orchestrator)

<function nli_checker at 0x7990e064c400>


In [34]:
print(nli_model)
print(nli_tokenizer)
print(embedder)
print(orchestrator)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
 

In [35]:
handpicked_problems = [
    {
        "problem": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?",
        "expected_answer": "$0.05",
        "targets": "numerical trap (intuitive wrong answer $0.10)"
    },
    {
        "problem": "A farmer has 17 sheep, and all but 9 die. How many sheep does the farmer have left?",
        "expected_answer": "9",
        "targets": "misleading wording trap"
    },
    {
        "problem": "If you're running a race and you pass the person in second place, what place are you in now?",
        "expected_answer": "second place",
        "targets": "misleading wording trap (common wrong answer: first place)"
    },
    {
        "problem": "A shirt originally costs $20. It goes on sale for 25% off, then the sale price is later increased by 25%. What is the final price?",
        "expected_answer": "$18.75",
        "targets": "numerical/sequential-percentage trap (common wrong answer: $20, assuming the discounts cancel out)"
    },
    {
        "problem": "Emily has 3 apples. She gives away 5 apples. How many apples does she have now, assuming she can borrow apples from a friend to give away and must pay them back later?",
        "expected_answer": "-2 (owes 2 apples)",
        "targets": "ambiguous/edge-case reasoning, tests whether Proposer or Critic handles negative-quantity logic sensibly"
    },
]

In [36]:
from datasets import load_dataset
import re

# Load GSM8K test dataset — note the namespaced repo id: "openai/gsm8k", not just "gsm8k"
gsm8k = load_dataset("openai/gsm8k", "main", split="test")

# Check one example first
print(gsm8k[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


In [37]:
import re

def extract_gsm8k_answer(answer_field: str) -> str:
    """
    GSM8K's answer field ends with '#### <number>'.
    Extract just that final number as a string.
    """

    match = re.search(r"####\s*([-+]?\d+(?:\.\d+)?)", answer_field)

    if match:
        return match.group(1)

    return ""

In [38]:
print(extract_gsm8k_answer(gsm8k[0]["answer"]))

18


In [39]:
gsm8k_problems = []

for i in range(12):  # first 12 examples from the test split
    example = gsm8k[i]
    gsm8k_problems.append({
        "problem": example["question"],
        "expected_answer": extract_gsm8k_answer(example["answer"]),
        "targets": "general reasoning (GSM8K)"
    })

print(len(gsm8k_problems))
print(gsm8k_problems[0])

12
{'problem': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'expected_answer': '18', 'targets': 'general reasoning (GSM8K)'}


In [40]:
benchmark_set = handpicked_problems + gsm8k_problems
print(len(benchmark_set))

17


In [41]:
def is_correct(extracted_conclusion: str, expected_answer: str) -> bool:
    """
    Returns True if expected_answer's value appears in the extracted_conclusion.
    Handle case differences and don't worry about being perfect — this is a
    heuristic scorer, not a formal proof checker.
    """
    conclusion = extracted_conclusion.lower().strip().replace("$", "")
    expected = expected_answer.lower().strip().replace("$", "")

    return expected in conclusion

In [42]:
print(is_correct("the ball costs $0.05", "$0.05"))
print(is_correct("the farmer has 9 sheep left", "9"))
print(is_correct("the answer is 20 dollars", "18"))

True
True
False


In [43]:
def single_prompt_baseline(problem: str) -> dict:
    """
    Run just ONE Proposer call, no Critic, no revision loop.
    Return a dict with at least: "conclusion" (extracted answer) and "raw_text".
    """
    proposer.reset()
    raw_text = proposer.respond(problem)
    conclusion = extract_conclusion(raw_text)
    return {
        "conclusion": conclusion,
        "raw_text": raw_text
    }

In [44]:
import time

results_log = []

for item in benchmark_set:
    problem = item["problem"]
    expected = item["expected_answer"]

    # --- Orchestrator (multi-agent) ---
    start = time.time()
    orch_result = orchestrator.run(problem)
    orch_time = time.time() - start

    orch_correct = is_correct(orch_result["final_answer"], expected)

    # --- Single-prompt baseline ---
    start = time.time()
    baseline_result = single_prompt_baseline(problem)
    baseline_time = time.time() - start

    baseline_correct = is_correct(baseline_result["conclusion"], expected)

    results_log.append({
        "problem": problem[:50] + "...",  # truncated for readability
        "targets": item["targets"],
        "expected": expected,
        "orch_answer": orch_result["final_answer"],
        "orch_correct": orch_correct,
        "orch_rounds": orch_result["rounds_used"],
        "orch_stopped_reason": orch_result["stopped_reason"],
        "orch_time_sec": round(orch_time, 1),
        "baseline_answer": baseline_result["conclusion"],
        "baseline_correct": baseline_correct,
        "baseline_time_sec": round(baseline_time, 1),
    })

    print(f"Done: {problem[:40]}... | Orch: {orch_correct} ({orch_result['rounds_used']} rounds) | Baseline: {baseline_correct}")

Done: A bat and a ball cost $1.10 in total. Th... | Orch: True (2 rounds) | Baseline: True
Done: A farmer has 17 sheep, and all but 9 die... | Orch: True (1 rounds) | Baseline: True
Done: If you're running a race and you pass th... | Orch: True (2 rounds) | Baseline: True
Done: A shirt originally costs $20. It goes on... | Orch: False (2 rounds) | Baseline: True
Done: Emily has 3 apples. She gives away 5 app... | Orch: False (1 rounds) | Baseline: False
Done: Janet’s ducks lay 16 eggs per day. She e... | Orch: True (3 rounds) | Baseline: True
Done: A robe takes 2 bolts of blue fiber and h... | Orch: True (1 rounds) | Baseline: True
Done: Josh decides to try flipping a house.  H... | Orch: False (2 rounds) | Baseline: False
Done: James decides to run 3 sprints 3 times a... | Orch: True (1 rounds) | Baseline: True
Done: Every day, Wendi feeds each of her chick... | Orch: True (1 rounds) | Baseline: True
Done: Kylar went to the store to buy glasses f... | Orch: True (1 rounds) | Baseline:

In [45]:
orch_correct_count = sum(1 for r in results_log if r["orch_correct"])
baseline_correct_count = sum(1 for r in results_log if r["baseline_correct"])
total = len(results_log)

avg_rounds = sum(r["orch_rounds"] for r in results_log) / total

print(f"Orchestrator accuracy: {orch_correct_count}/{total} ({100*orch_correct_count/total:.1f}%)")
print(f"Baseline accuracy:     {baseline_correct_count}/{total} ({100*baseline_correct_count/total:.1f}%)")
print(f"Average rounds used:   {avg_rounds:.2f}")

Orchestrator accuracy: 12/17 (70.6%)
Baseline accuracy:     13/17 (76.5%)
Average rounds used:   1.65


In [46]:
run1_results = [  # yesterday's run, reconstructed from what you reported: 10/17 orch, 14/17 baseline, avg 1.94 rounds
    # (we don't have the full per-problem detail saved from yesterday, only the aggregate — note this as a limitation)
]

run2_results = results_log.copy()  # save today's run before the next one overwrites results_log

print(len(run2_results))

17


In [47]:
results_log = []

for item in benchmark_set:
    problem = item["problem"]
    expected = item["expected_answer"]

    start = time.time()
    orch_result = orchestrator.run(problem)
    orch_time = time.time() - start
    orch_correct = is_correct(orch_result["final_answer"], expected)

    start = time.time()
    baseline_result = single_prompt_baseline(problem)
    baseline_time = time.time() - start
    baseline_correct = is_correct(baseline_result["conclusion"], expected)

    results_log.append({
        "problem": problem[:50] + "...", "targets": item["targets"], "expected": expected,
        "orch_answer": orch_result["final_answer"], "orch_correct": orch_correct,
        "orch_rounds": orch_result["rounds_used"], "orch_stopped_reason": orch_result["stopped_reason"],
        "orch_time_sec": round(orch_time, 1),
        "baseline_answer": baseline_result["conclusion"], "baseline_correct": baseline_correct,
        "baseline_time_sec": round(baseline_time, 1),
    })
    print(f"Done: {problem[:40]}... | Orch: {orch_correct} ({orch_result['rounds_used']} rounds) | Baseline: {baseline_correct}")

run3_results = results_log.copy()
print(len(run3_results))

Done: A bat and a ball cost $1.10 in total. Th... | Orch: True (1 rounds) | Baseline: True
Done: A farmer has 17 sheep, and all but 9 die... | Orch: True (1 rounds) | Baseline: True
Done: If you're running a race and you pass th... | Orch: True (2 rounds) | Baseline: False
Done: A shirt originally costs $20. It goes on... | Orch: True (1 rounds) | Baseline: True
Done: Emily has 3 apples. She gives away 5 app... | Orch: False (3 rounds) | Baseline: False
Done: Janet’s ducks lay 16 eggs per day. She e... | Orch: True (2 rounds) | Baseline: True
Done: A robe takes 2 bolts of blue fiber and h... | Orch: True (1 rounds) | Baseline: True
Done: Josh decides to try flipping a house.  H... | Orch: False (3 rounds) | Baseline: False
Done: James decides to run 3 sprints 3 times a... | Orch: True (1 rounds) | Baseline: True
Done: Every day, Wendi feeds each of her chick... | Orch: True (2 rounds) | Baseline: True
Done: Kylar went to the store to buy glasses f... | Orch: True (1 rounds) | Baseline:

In [48]:
print(engine)
print(orchestrator)
print(len(benchmark_set))

17


In [49]:
result_emily = orchestrator.run(
    "Emily has 3 apples. She gives away 5 apples. How many apples does she have now, "
    "assuming she can borrow apples from a friend to give away and must pay them back later?"
)
print(result_emily["final_answer"], "|", result_emily["stopped_reason"])
for entry in result_emily["trace"]:
    print(entry["proposer_text"])
    print("---")

emily has 0 apples, owes her friend apples | approved
Emily starts with 3 apples. If she gives away 5 apples and can borrow from a friend, she would end up with 0 apples because she cannot give away more apples than she has. She would owe her friend apples until she can pay them back.

FINAL ANSWER: Emily has 0 apples, owes her friend apples.
---


In [50]:
result_bat = orchestrator.run(
    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?"
)
print(result_bat["final_answer"], "|", result_bat["stopped_reason"])
for entry in result_bat["trace"]:
    print(entry["proposer_text"])
    print("CRITIC:", entry.get("critic_text", "N/A") if "critic_text" in entry else "(not stored)")
    print("---")

$0.05 | cycle_detected
Let's denote the cost of the ball as x. According to the problem, the bat costs $1.00 more than the ball. Therefore, the cost of the bat can be expressed as x + $1.00.

The total cost of the bat and the ball is given as $1.10. So, we can set up the following equation:

x (ball's cost) + (x + $1.00) (bat's cost) = $1.10 (total cost)

Solving this equation, we get:

2x + $1.00 = $1.10

Subtracting $1.00 from both sides gives:

2x = $0.10

Finally, dividing by 2 yields:

x = $0.05

So, the cost of the ball is $0.05.

FINAL ANSWER: $0.05
CRITIC: (not stored)
---
Thank you for your feedback. I understand the confusion in my previous answer. Let's re-analyze the problem.

Let's denote the cost of the ball as x. The bat, which costs $1.00 more than the ball, can be expressed as x + $1.00.

The total cost of the bat and the ball is $1.10. So we can write the equation as follows:

x (ball's cost) + (x + $1.00) (bat's cost) = $1.10 (total cost)

Solving this equation gives

In [51]:
print(run2_results[0].keys())

dict_keys(['problem', 'targets', 'expected', 'orch_answer', 'orch_correct', 'orch_rounds', 'orch_stopped_reason', 'orch_time_sec', 'baseline_answer', 'baseline_correct', 'baseline_time_sec'])


In [52]:
def compute_tier_metrics(results, hard_max_rounds=5):
    """
    Split results into hand-picked vs GSM8K tiers using entry["targets"],
    then compute:
        - MCR: Mean Consensus Rounds
        - RCR: Round Compression Ratio
        - Average orchestration time
        - Accuracy

    Returns:
        {
            "handpicked": {...},
            "gsm8k": {...}
        }
    """

    handpicked = [
        entry for entry in results
        if entry["targets"] != "general reasoning (GSM8K)"
    ]

    gsm8k = [
        entry for entry in results
        if entry["targets"] == "general reasoning (GSM8K)"
    ]

    def calculate_metrics(entries):

        if len(entries) == 0:
            return {
                "MCR": 0.0,
                "RCR": 0.0,
                "avg_time_sec": 0.0,
                "accuracy": 0.0
            }

        mcr = sum(
            entry["orch_rounds"]
            for entry in entries
        ) / len(entries)

        rcr = sum(
            entry["orch_rounds"] / hard_max_rounds
            for entry in entries
        ) / len(entries)

        avg_time = sum(
            entry["orch_time_sec"]
            for entry in entries
        ) / len(entries)

        accuracy = sum(
            1 for entry in entries
            if entry["orch_correct"]    # <-- FIXED: was "is_correct", your data uses "orch_correct"
        ) / len(entries)

        return {
            "MCR": mcr,
            "RCR": rcr,
            "avg_time_sec": avg_time,
            "accuracy": accuracy
        }

    return {
        "handpicked": calculate_metrics(handpicked),
        "gsm8k": calculate_metrics(gsm8k)
    }

In [53]:
run2_tier_metrics = compute_tier_metrics(run2_results, hard_max_rounds=5)

print("RUN 2")
print("======")
print("\nHand-picked:")
print(run2_tier_metrics["handpicked"])
print("\nGSM8K:")
print(run2_tier_metrics["gsm8k"])

RUN 2

Hand-picked:
{'MCR': 1.6, 'RCR': 0.32, 'avg_time_sec': 31.26, 'accuracy': 0.6}

GSM8K:
{'MCR': 1.6666666666666667, 'RCR': 0.3333333333333333, 'avg_time_sec': 35.34166666666667, 'accuracy': 0.75}


In [54]:
run3_tier_metrics = compute_tier_metrics(run3_results, hard_max_rounds=5)

print("RUN 3")
print("======")
print("\nHand-picked:")
print(run3_tier_metrics["handpicked"])
print("\nGSM8K:")
print(run3_tier_metrics["gsm8k"])

RUN 3

Hand-picked:
{'MCR': 1.6, 'RCR': 0.32, 'avg_time_sec': 18.14, 'accuracy': 0.8}

GSM8K:
{'MCR': 1.8333333333333333, 'RCR': 0.3666666666666667, 'avg_time_sec': 59.574999999999996, 'accuracy': 0.75}


In [55]:
print("PHASE 4 TIER METRICS")
print("====================")

for run_name, metrics in [("Run 2", run2_tier_metrics), ("Run 3", run3_tier_metrics)]:
    print(f"\n{run_name}")
    for tier in ["handpicked", "gsm8k"]:
        m = metrics[tier]
        print(f"\n{tier.upper()}")
        print(f"MCR:            {m['MCR']:.3f}")
        print(f"RCR:            {m['RCR']:.3f}")
        print(f"Average time:   {m['avg_time_sec']:.3f} sec")
        print(f"Accuracy:       {m['accuracy']:.3f}")

PHASE 4 TIER METRICS

Run 2

HANDPICKED
MCR:            1.600
RCR:            0.320
Average time:   31.260 sec
Accuracy:       0.600

GSM8K
MCR:            1.667
RCR:            0.333
Average time:   35.342 sec
Accuracy:       0.750

Run 3

HANDPICKED
MCR:            1.600
RCR:            0.320
Average time:   18.140 sec
Accuracy:       0.800

GSM8K
MCR:            1.833
RCR:            0.367
Average time:   59.575 sec
Accuracy:       0.750
